# 02 – Data Preparation

This notebook follows the CRISP-DM data preparation phase.  
The purpose is to transform the raw dataset into a clean and structured dataset suitable for analysis and modelling.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data_raw/transit_2026_raw.csv")

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.shape
df.head()

In [ ]:
df.columns

## 1. Select data

The data selection is based on the criteria established during the initial data collection phase (Section 2.1).

The dataset consists of tracking data for volumetric pumps over a defined time period (January 2026). All available observations are included, as the dataset size is manageable and no sampling is required.

Variable selection is guided by relevance to the analytical objective, which focuses on understanding movement behaviour. Consequently, variables related to movement (e.g., distance, duration, segment count), time (e.g., day of week, timestamp), and location (e.g., floor and room) are retained.

Variables that do not provide analytical value are excluded. In particular, global location variables are removed, as the data understanding phase showed that they provide identical granularity to local variables. Similarly, classification variables are excluded, as the dataset contains only a single equipment type.

No additional data sources are included, as all required information is available within the dataset. The selected data is therefore considered sufficient to support the analysis.

In [ ]:
cols_to_keep = [
    # ID
    "object_key",

    # Movement variables (KERNE)
    "distance_mtr",
    "duration_sec",
    "stay_duration_sec",
    "segment_count",
    "distance_floor",

    # Time (keep raw → vi konstruerer senere)
    "sd_date_string",
    "st_time",
    "sd_day_of_week_name",

    # Location (keep local + start only)
    "sl_floor_local_name",
    "sl_room_local_name"
]

df = df[cols_to_keep]

In [ ]:
print("Shape after selection:", df.shape)

## 2. Clean data

The data cleaning process is based on the data quality assessment conducted in Section 8 (Verify Data Quality), which evaluated the dataset in terms of completeness, correctness, and consistency.

Regarding completeness, no missing values are observed in key analytical variables such as distance, duration, and stay duration. Missing values are primarily confined to location-related variables, particularly at the room level, where more than 50% of observations are missing. As identified in the data understanding phase, this missingness is systematic and reflects limitations in location tracking rather than data errors. Consequently, missing location values are retained.

In terms of correctness, all numerical variables contain valid values, with no indication of impossible values such as negative distances or durations. Although the distributions exhibit strong right-skewness and include extreme values, these are considered plausible and reflect real variation in movement behaviour. Therefore, no outlier removal is performed.

Consistency checks further confirm that the dataset is internally coherent. No duplicate records are identified, and temporal ordering is valid, with no cases where end time precedes start time. Observed inconsistencies between distance and location variables (e.g., non-zero distance within the same location or zero distance between different locations) are interpreted as characteristics of the measurement process rather than data errors.

Overall, no substantial cleaning operations are required. The dataset is considered sufficiently clean and reliable for further analysis, and only minimal preprocessing is applied.

### Missing data

In [ ]:
# Overview of missing values
df.isna().sum().sort_values(ascending=False)

In [ ]:
import missingno as msno
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

msno.matrix(
    df,
    sparkline=False,
    color=(0.27, 0.0, 0.33)  # lilla (observed)
)

plt.title("Missing data pattern", fontsize=30)
plt.show()

Missing values are primarily present in location variables, particularly 
sl_room_local_name (54%) and to a lesser extent sl_floor_local_name (~4%).

As missingness is structural and key movement variables contain no missing values, 
all variables are retained.

In [ ]:
!pip install missingno

### Temporal consistency

Temporal variables are stored as strings and converted to appropriate datetime formats. 
Time variables are recorded at minute-level granularity and are transformed accordingly.

In [ ]:
# Convert date variables
df["sd_date"] = pd.to_datetime(df["sd_date_string"], format="%d-%m-%Y")

In [ ]:
# Convert time variables (HH.MM.SS → HH:MM:SS)
df["st_time"] = pd.to_datetime(df["st_time"], format="%H.%M.%S").dt.time

## 3. Construct Data

In this step, derived attributes are constructed to better capture temporal patterns and movement behaviour.

The feature construction is guided by findings from the Data Understanding phase, including:
- Strong temporal variation in movement activity  
- Highly skewed movement distributions  
- Significant differences between high- and low-activity devices  

All transformations are implemented in the notebook to ensure reproducibility.

In [ ]:
df_constructed = df.copy()

### Temporal features

To capture temporal patterns in movement activity, time-based features are derived from the start time.

Previous analysis showed that movements are concentrated during daytime hours, with lower activity at night.

In [ ]:
# Hour of day
df_constructed["hour"] = df_constructed["st_time"].apply(lambda x: x.hour)
# Time period (aligned with analysis: day vs night)
df_constructed["time_period"] = df_constructed["hour"].apply(
    lambda x: "day" if 7 <= x < 19 else "night"
)

### Movement behaviour features

To better represent movement characteristics, new variables are constructed based on distance, duration, and stay duration.

The Data Understanding phase showed that devices spend most of their time stationary and that movement-related variables are highly skewed.

**Movement ratio** is constructed to capture the proportion of time spent moving relative to total time.

This directly reflects the observed pattern that devices are stationary for most of the time.

In [ ]:
df_constructed["movement_ratio"] = (
    df_constructed["duration_sec"] /
    (df_constructed["duration_sec"] + df_constructed["stay_duration_sec"])
).fillna(0)

<!-- **Average speed** is calculated to capture movement efficiency across observations. -->

**Average speed** is calculated to capture movement efficiency across observations.

In [ ]:
df_constructed["avg_speed"] = (
    df_constructed["distance_mtr"] / df_constructed["duration_sec"]
)

df_constructed["avg_speed"] = df_constructed["avg_speed"].replace([float("inf")], 0)

To account for skewed distributions observed in the data, selected variables are transformed into more interpretable forms.

In [ ]:
# Log transform (optional but aligned with analysis)
df_constructed["log_distance"] = np.log1p(df_constructed["distance_mtr"])

### Activity level

To capture differences in device usage, an activity-level variable is constructed based on movement frequency.

The Data Understanding phase showed that a small number of devices account for a large share of movements.

In [ ]:
activity = (
    df_constructed.groupby("object_key")
    .size()
    .reset_index(name="n_movements")
)

threshold = activity["n_movements"].median()

activity["activity_level"] = activity["n_movements"].apply(
    lambda x: "high" if x >= threshold else "low"
)

df_constructed = df_constructed.merge(
    activity[["object_key", "activity_level"]],
    on="object_key",
    how="left"
)

### Feature construction conclusion
The dataset now includes derived attributes reflecting temporal patterns and movement behaviour and is ready for further analysis and modelling.

In [ ]:
df_constructed.head()

## 4. Integrate Data

Data integration is performed as part of the feature construction process, where device-level information (movement frequency) is aggregated and merged back into the dataset.

As the dataset originates from a single source, no additional external data sources are integrated.

The dataset therefore already contains both event-level and aggregated device-level information.

The dataset now integrates both movement-level and device-level information and is ready for further processing.

In [ ]:
df_constructed.head()

## 5. Format data

In this step, the dataset is formatted to ensure compatibility with subsequent analysis and modelling.

Formatting includes minor syntactic adjustments such as data types and column structure. These transformations do not change the meaning of the data.

## 6. Save data processed data

In [ ]:
df_constructed.to_csv("../data_processed/transit_2026_prepared.csv", index=False)